# Search with SyftHub, cell by cell

The Python SDK in one sitting: browse the catalogue, compose a search over several publishers,
see what each one will cost and need **before** anything is sent, resolve what blocks, run it, and
read the results. Every query goes from this process straight to the publisher's Space; the Hub
only serves the catalogue, the per-Space tokens and, for metered sources, the payment credential.

This notebook runs against `syfthub.testing.FakeHub`, an in-memory Hub with three publishers, so
it works offline. To run it against a real Hub, replace the first cell with

```python
hub = syfthub.Hub(token="syft_pat_...", budget=1.00)   # or SYFTHUB_URL / SYFTHUB_TOKEN
```

and use your own source paths. The sync twin `syfthub.Hub` is used throughout because notebooks
already run an event loop; `syfthub.AsyncHub` is the same API with `await`.

In [1]:
import syfthub
from syfthub.testing import FakeHub

fake = FakeHub()
fake.add_source("alice", "notes", description="parental leave policy", tags=["hr"])
fake.add_source(
    "erin", "trials", description="phase 3 trial reports", price=0.05
)  # prepaid, per request
fake.add_source(
    "kim", "metered", description="metered corpus", mpp_price=0.02
)  # pay-per-request (MPP)
fake.add_source("kim", "assistant", description="a model that summarises", type="model")
fake.space("alice").reply(
    "notes", documents=[{"content": "Parental leave is 16 weeks at full pay.", "score": 0.82}]
)
fake.space("erin").reply(
    "trials",
    documents=[{"content": "Phase 3 reported 3 serious adverse events.", "score": 0.74}],
    cost=0.05,
)
fake.space("kim").reply(
    "metered",
    documents=[{"content": "Adverse events were mild and transient.", "score": 0.61}],
    mpp_price=0.02,
)
fake.space("kim").reply(
    "assistant", summary="Most phase 3 adverse events were mild; three were serious."
)

hub = syfthub.Hub(fake.url, token="demo-token", budget=1.00, options=fake.options())
hub

Hub('https://hub.test', as=token)

## 1. Browse the catalogue

A source is a published endpoint, addressed as `owner/slug`. `hub.sources.list()` reads every page;
`filter`, `matching` and `pick` narrow a listing without another request.

In [2]:
listing = hub.sources.list()
for s in listing:
    price = (
        f"{s.pricing.price} per {s.pricing.unit.value} via {s.pricing.rail.value}"
        if s.pricing
        else "free"
    )
    print(f"{s.path:15} {s.type.value:18} {s.health.value:8} {price}")

alice/notes     data_source        healthy  free
erin/trials     data_source        healthy  0.05 USD per request via xendit
kim/metered     data_source        healthy  0.02 USD per request via mpp
kim/assistant   model              healthy  free


In [3]:
(
    listing.filter(free=True).paths,
    listing.filter(type="model").paths,
    listing.matching("adverse").paths,
)

(('alice/notes', 'kim/assistant'), ('kim/assistant',), ())

In [4]:
source = hub.sources.get("erin/trials")
source.pricing

Pricing(rail=<Rail.XENDIT: 'xendit'>, prices=(Price(amount=Money(amount=Decimal('0.05'), currency='USD'), unit=<PriceUnit.REQUEST: 'request'>, policy_type='xendit'),), currency='USD', applied_to=(), wallet_id='w-erin', wallet_owner='erin', bundles=(Bundle(id='starter', amount=Money(amount=Decimal('1'), currency='USD'), credits=None),), payment_url='https://erin.space.test/api/v1/payments', credits_url='https://erin.space.test/api/v1/credits', invoices_url='https://erin.space.test/api/v1/invoices')

## 2. Compose a plan

`hub.search` returns a `SearchPlan`. Nothing is sent yet: the plan is a frozen value you can inspect,
save, and change with `skip`.

In [5]:
plan = hub.search("adverse events in phase 3", listing, limit=3, similarity_threshold=0.5)
plan

SearchPlan(query='adverse events in phase 3', sources=('alice/notes', 'erin/trials', 'kim/metered', 'kim/assistant'), limit=3, similarity_threshold=0.5, include_metadata=True, skips=(), max_sources=10, version='2.0.0a1', created_at=datetime.datetime(2026, 10, 12, 9, 42, 18, 282920, tzinfo=datetime.timezone.utc))

## 3. Pre-flight

One row per source with a verdict, and one action per *cause*. `pre.ready` is `True` only when
every row will be sent. Here `erin/trials` bills a prepaid wallet that is empty, so it is held and a
`TopUp` tells you the shortfall and the bundle that covers it.

In [6]:
pre = plan.preflight()
for row in pre:
    print(f"{row.path:15} {row.verdict.value:16} {row.estimate!s:10} {row.note or ''}")
print("ready:", pre.ready, "| estimate for the sending rows:", pre.estimate)

alice/notes     ready            0 USD      
erin/trials     needs_credits    0.05 USD   wallet erin/xendit/USD holds 0 USD, this plan needs 0.05 USD
kim/metered     will_ask_to_pay  0.02 USD   MPP: the Space issues the charge
kim/assistant   ready            0 USD      
ready: False | estimate for the sending rows: {'USD': Money(amount=Decimal('0.02'), currency='USD')}


In [7]:
for action in pre.actions:
    print(action.kind.value, "affects", action.paths)
    for step in action.steps:
        print("  -", step)

top_up affects ('erin/trials',)
  - await action.top_up() buys the starter bundle (1 USD) on wallet erin/xendit/USD
  - pay at invoice.checkout_url, then await invoice.wait_paid()
  - plan.execute() again: pre-flight reads the new balance


## 4. Resolve what blocks

Three verbs cover every decision: `plan.skip(...)`, `hub.set_budget(...)`, and for a `TopUp`,
`action.top_up()`. A top-up returns an `Invoice` with a checkout URL; nothing is charged until the
user pays it. The fake Space lets us pay it here.

In [8]:
(topup,) = pre.actions
invoice = topup.top_up()  # the suggested bundle
print(invoice.status.value, invoice.amount, invoice.checkout_url)

fake.space("erin").pay(invoice.id)  # the user completes the checkout
invoice = invoice.refresh()
print(invoice.status.value, "| wallet balance:", topup.wallet.refresh().balance)

pending 1 USD https://erin.space.test/checkout/1
paid | wallet balance: 1 USD


In [9]:
pre = plan.preflight()  # pre-flight again: the paid wallet now passes
[(row.path, row.verdict.value) for row in pre], pre.ready

([('alice/notes', 'ready'),
  ('erin/trials', 'ready'),
  ('kim/metered', 'will_ask_to_pay'),
  ('kim/assistant', 'ready')],
 True)

## 5. Execute

One request per Space, in parallel. `results.chunks` ranks scored passages across the document
sources; `results.answers` holds what model sources returned (`results.documents` is both, chunks
first). The metered source was paid on the way: the
Space challenged, the Hub signed the credential, and the charge shows in `results.cost`.

In [10]:
results = plan.execute()
for path, doc in results.chunks:
    print(f"[{doc.score:.2f}] {path:12} {doc.content}")
print()
print("answers:", results.answers)
print(
    "cost:",
    results.cost,
    "| spent this session:",
    hub.spent,
    "| budget left:",
    hub.budget.remaining,
)

[0.82] alice/notes  Parental leave is 16 weeks at full pay.
[0.74] erin/trials  Phase 3 reported 3 serious adverse events.
[0.61] kim/metered  Adverse events were mild and transient.

answers: {'kim/assistant': 'Most phase 3 adverse events were mild; three were serious.'}
cost: {'USD': Money(amount=Decimal('0.07'), currency='USD')} | spent this session: {'USD': Money(amount=Decimal('0.07'), currency='USD')} | budget left: {'USD': Money(amount=Decimal('0.93'), currency='USD')}


In [11]:
for r in results:
    print(
        f"{r.path:15} {r.outcome.value:10} {r.reason.value if r.reason else '':14} "
        f"paid_from={r.paid_from or '-':18} cost={r.cost or '-'}"
    )

alice/notes     returned                  paid_from=-                  cost=-
erin/trials     returned                  paid_from=erin/xendit/USD    cost=0.05 USD
kim/metered     returned                  paid_from=-                  cost=0.02 USD
kim/assistant   returned                  paid_from=-                  cost=-


## 6. When a Space says no

Rejections and failures never raise at execute time. They land in the row with a `Reason`, and
`results.retryable` tells you which ones are worth sending again.

In [12]:
fake.space("alice").reject(
    "notes",
    outcome="rate_limited",
    details={"retry_after_seconds": 9},
    headers={"Retry-After": "9"},
)

results = plan.execute()
limited = results["alice/notes"]
print(
    limited.outcome.value, limited.reason.value, "| reset in", limited.rate_limit.reset_seconds, "s"
)
print("ok:", [r.path for r in results.ok])
print("retryable:", [r.path for r in results.retryable])

rejected rate_limited | reset in 9 s
ok: ['erin/trials', 'kim/metered', 'kim/assistant']
retryable: ['alice/notes']


In [13]:
fake.space("alice").reply(
    "notes", documents=[{"content": "Parental leave is 16 weeks at full pay.", "score": 0.82}]
)
results = results.retry()  # re-sends only the retryable rows and merges them in
[(r.path, r.outcome.value) for r in results]

[('alice/notes', 'returned'),
 ('erin/trials', 'returned'),
 ('kim/metered', 'returned'),
 ('kim/assistant', 'returned')]

## 7. Budget

There is no budget until you set one. A budget is one cap per currency, since publishers bill in
their own currencies and the SDK never converts money: `budget=1.00` caps USD only, and
`budget={"USD": 1.00, "IDR": 50000}` caps both. Rows whose estimate would cross the cap in their
currency are held with an `OverBudget` action, in plan order: the first row that crosses and every
paid row in that currency after it. A currency with no cap is not budgeted. `execute(strict=True)`
raises `ActionRequired` instead of sending partially.

In [14]:
print(hub.budget)
hub.set_budget({"USD": 0.20})  # 0.06 left: the first row fits, the second would cross
pre = hub.search("adverse events", ["erin/trials", "kim/metered"]).preflight()
print([(row.path, row.verdict.value) for row in pre])
print(
    [a.kind.value for a in pre.actions],
    "estimate",
    pre.actions[0].estimate,
    "remaining",
    pre.actions[0].budget.remaining,
)

limits={'USD': Money(amount=Decimal('1.0'), currency='USD')} spent={'USD': Money(amount=Decimal('0.14'), currency='USD')}
[('erin/trials', 'ready'), ('kim/metered', 'over_budget')]
['over_budget'] estimate 0.07 USD remaining {'USD': Money(amount=Decimal('0.06'), currency='USD')}


In [15]:
try:
    hub.search("adverse events", ["erin/trials", "kim/metered"]).execute(strict=True)
except syfthub.ActionRequired as e:
    print("ActionRequired:", e)
hub.set_budget(None)  # no cap

ActionRequired: 1 action(s) required before sending: over_budget


Budget(limits={}, spent={'USD': Money(amount=Decimal('0.14'), currency='USD')})

## 8. Save and resume

Plans, invoices and results are plain data. An app can hand a top-up to the user, save the job, and
pick it up later or in another process. `hub.load` rebinds a snapshot on the sync twin;
`SearchPlan.from_dict(hub, d)` does the same on `AsyncHub`.

In [16]:
import json

saved = json.dumps({"plan": plan.to_dict(), "invoice": invoice.to_dict()})
restored = hub.load(syfthub.SearchPlan, json.loads(saved)["plan"])
restored == plan, restored.execute().cost

(True, {'USD': Money(amount=Decimal('0.07'), currency='USD')})

## 9. What the SDK sent

The fake records every request, with secrets redacted. Note that the queries went to the Spaces
(`alice.space.test`, `erin.space.test`, `kim.space.test`) and the Hub only saw catalogue, token and
pay calls.

In [17]:
from collections import Counter

Counter((c.host, c.path.split("/api/v1/")[-1].split("?")[0]) for c in fake.calls).most_common()

[(('erin.space.test', 'credits'), 8),
 (('kim.space.test', 'endpoints/metered/query'), 6),
 (('alice.space.test', 'endpoints/notes/query'), 4),
 (('hub.test', 'token'), 3),
 (('erin.space.test', 'endpoints/trials/query'), 3),
 (('hub.test', 'wallet/pay'), 3),
 (('kim.space.test', 'endpoints/assistant/query'), 3),
 (('hub.test', 'endpoints/public'), 1),
 (('hub.test', 'auth/me'), 1),
 (('erin.space.test', 'payments'), 1),
 (('erin.space.test', 'invoices'), 1)]

In [18]:
hub.close()

## Where next

- `examples/openai_chat.py` is the reference app: pre-flight, hand the top-up to the user, resume after payment, compose an answer with OpenAI.
- `docs/guides/python-sdk.md` is the full guide, including the migration table from the 0.x client.
- `syfthub.testing.FakeHub` is what this notebook ran on; use it in your own tests.